In [ ]:
#| default_exp sanskrit

In [ ]:
#| hide
from nbdev.showdoc import *

## Why every store gets this

Before the fold, a Sanskrit query found nothing:

| query | against | hits |
|---|---|---|
| `देवी` | `दे॒वी`, the same word accented | 0 |
| `श्रीमाता` | `śrīmātā`, the same word in IAST | 0 |
| `kuṇḍa` | `cidagnikuṇḍasambhūtā` | 0 |

The fix is to fold both scripts to one lossy ASCII key at index time, rather than transliterate at
query time. `simplify casefold` already turns IAST `śrīmātā` into `srimata`; `deva2ascii` is
chosen so Devanagari `श्रीमाता` lands on `srimata` too.

This is on for every store, not only Sanskrit ones, because it costs nothing on English text and
it is the largest measured retrieval win in the repository. Everything else about Sanskrit, metre,
verse chunking, citations and lemmas, is in [ganapati](https://github.com/vedicreader/ganapati).

In [ ]:
#| export
import re, unicodedata
from functools import lru_cache
from fastcore.all import L, Path, ifnone

VEDIC_MARKS = re.compile('['
    '॑-॔'      # udatta, anudatta, devanagari grave/acute
    '᳐-᳹'      # vedic extensions: tone marks, nasalisations
    '꣠-꣰'      # devanagari extended combining digits, stopping before the candrabindu block
    ']')

DANDA, DDANDA = '।', '॥'          # । ॥

_INDEP = {'अ':'a','आ':'a','इ':'i','ई':'i','उ':'u','ऊ':'u','ऋ':'r','ॠ':'r','ऌ':'l','ॡ':'l',
          'ए':'e','ऐ':'ai','ओ':'o','औ':'au','ऍ':'e','ऎ':'e','ऑ':'o','ऒ':'o'}
_CONS  = {'क':'k','ख':'kh','ग':'g','घ':'gh','ङ':'n','च':'c','छ':'ch','ज':'j','झ':'jh','ञ':'n',
          'ट':'t','ठ':'th','ड':'d','ढ':'dh','ण':'n','त':'t','थ':'th','द':'d','ध':'dh','न':'n',
          'प':'p','फ':'ph','ब':'b','भ':'bh','म':'m','य':'y','र':'r','ल':'l','व':'v',
          'श':'s','ष':'s','स':'s','ह':'h','ळ':'l','ऴ':'l','ऱ':'r','ऩ':'n',
          'क़':'k','ख़':'kh','ग़':'g','ज़':'z','ड़':'d','ढ़':'dh','फ़':'f','य़':'y'}
_MATRA = {'ा':'a','ि':'i','ी':'i','ु':'u','ू':'u','ृ':'r','ॄ':'r','ॢ':'l','ॣ':'l',
          'े':'e','ै':'ai','ो':'o','ौ':'au','ॅ':'e','ॆ':'e','ॉ':'o','ॊ':'o'}
# `ꣳ`/`ꣲ`/`꣰` are the Vedic candrabindu signs — a nasal, not an accent, so they fold to `m` and
# line up with the `ṃ` an IAST edition of the same line prints.
_SIGN  = {'ं':'m','ः':'h','ँ':'m','ऽ':'', '़':'', 'ꣲ':'m','ꣳ':'m','ꣴ':'m','ꣵ':'m','ꣶ':'m','ꣷ':'m'}
VIRAMA = '्'
DEVA_DIGITS = {chr(0x966+i): str(i) for i in range(10)}

def strip_vedic(s:str) -> str:
    'Drop Vedic tone marks from Devanagari, leaving Latin diacritics alone. NFC out.'
    if not s: return ''
    return unicodedata.normalize('NFC', VEDIC_MARKS.sub('', unicodedata.normalize('NFD', s)))

def deva2ascii(s:str) -> str:
    'Devanagari to a bare ASCII key, applying the implicit `a` of an unmarked consonant.'
    out, i, n = [], 0, len(s)
    while i < n:
        ch = s[i]
        if ch in _CONS:
            out.append(_CONS[ch]); i += 1
            # look past a nukta that was not pre-composed
            while i < n and s[i] == '़': i += 1
            if i < n and s[i] == VIRAMA: i += 1; continue          # bare consonant, no vowel
            if i < n and s[i] in _MATRA: out.append(_MATRA[s[i]]); i += 1; continue
            out.append('a'); continue                                # implicit vowel
        if ch in _INDEP: out.append(_INDEP[ch]); i += 1; continue
        if ch in _SIGN:  out.append(_SIGN[ch]);  i += 1; continue
        if ch in _MATRA: out.append(_MATRA[ch]); i += 1; continue    # orphan matra
        if ch in DEVA_DIGITS: out.append(DEVA_DIGITS[ch]); i += 1; continue
        if ch == VIRAMA: i += 1; continue
        out.append(ch); i += 1
    return ''.join(out)

def _latn_fold(s:str) -> str:
    'IAST/ISO to ASCII — the same folding `simplify casefold` already applies to a Latin token.'
    d = unicodedata.normalize('NFD', s)
    return ''.join(c for c in d if unicodedata.category(c) != 'Mn').lower()

DEVANAGARI = re.compile('[ऀ-ॿ᳐-᳹꣠-ꣿ]')

def detect_script(s:str) -> str:
    "`'deva'`, `'latn'` or `'other'` — enough to pick a folding path."
    if not s: return 'other'
    d = len(DEVANAGARI.findall(s))
    l = sum(1 for c in s if 'a' <= c.lower() <= 'z' or unicodedata.category(c) == 'Ll')
    if d and d >= l: return 'deva'
    return 'latn' if l else 'other'

@lru_cache(maxsize=1<<16)
def fold_token(s:str) -> str:
    '''The ASCII index key for one token, whatever script it arrived in.

    Cached because it is called once per token per FTS write *and* per query, and a corpus is a
    small vocabulary repeated: 70,413 tokens of the eval corpus are 4,153 distinct strings, so
    94% of the calls are asking a question that has already been answered. It is three unicode
    normalisations and a per-character category scan deep, which is why it was a third of the cost
    of indexing a corpus with no Sanskrit in it at all. Pure function of `s`, so the cache is only
    ever a memo — same output, 5.5x fewer of them computed.'''
    s = strip_vedic(s)
    return _latn_fold(deva2ascii(s) if DEVANAGARI.search(s) else s)

In [ ]:
#| hide
# the whole point: the same word in either script folds to one key
for deva, iast in [('श्रीमाता','śrīmātā'), ('देवी','devī'), ('दे॒वी','devī'),
                   ('चिदग्निकुण्डसम्भूता','cidagnikuṇḍasambhūtā'), ('गणपतिꣳ','gaṇapatiṃ')]:
    assert fold_token(deva) == fold_token(iast), (deva, fold_token(deva), iast, fold_token(iast))
assert fold_token('श्रीमाता') == 'srimata'
# accents fold away; they are the difference between 0 hits and 1
assert fold_token('दे॒वी') == fold_token('देवी') == 'devi'
assert strip_vedic('सर॑स्वती॒') == 'सरस्वती'
# strip_vedic must leave Latin diacritics alone, since folding those is _latn_fold's job
assert strip_vedic('śrīmātā') == 'śrīmātā'
assert detect_script('श्रीमाता') == 'deva' and detect_script('śrīmātā') == 'latn'
assert detect_script('') == 'other'
# the implicit `a` of an unmarked consonant, and virama suppressing it
assert deva2ascii('क') == 'ka' and deva2ascii('क्') == 'k' and deva2ascii('का') == 'ka'

## The tokenizer

The fold could have been a second indexed column, the way an application usually does it. As an
FTS5 tokenizer instead it needs no schema change, no second column and no query rewriting, and it
works for any store that already exists.

`sanskrit_tokenizer` wraps another tokenizer rather than replacing it. It emits the fold as a
colocated token inside porter, so English keeps its stemming and identifiers stay whole.

In [ ]:
#| export
SANSKRIT_TOKENIZE = 'sanskrit'

def sanskrit_tokenizer(con, args):
    'A *wrapping* FTS5 tokenizer that folds Sanskrit so a query matches regardless of script.'
    import apsw.fts5
    rest = [a for a in args if '=' not in a]
    opts = dict(a.split('=', 1) for a in args if '=' in a)
    keep_orig = opts.get('original', '1') != '0'
    inner = con.fts5_tokenizer(rest[0], rest[1:]) if rest else apsw.fts5.UnicodeWordsTokenizer(con, [])
    def tok(utf8, flags, locale):
        for start, end, *toks in inner(utf8, flags, locale):
            out = []
            for t in toks:
                if keep_orig and t not in out: out.append(t)
                if (f := fold_token(t)) and f not in out: out.append(f)
            if out: yield (start, end, *out)
    return tok

def register_sanskrit(db):
    "Register the `sanskrit` FTS5 tokenizer on a connection. Idempotent."
    conn = getattr(db, 'conn', db)
    try: conn.register_fts5_tokenizer(SANSKRIT_TOKENIZE, sanskrit_tokenizer)
    except Exception: pass
    return db

In [ ]:
#| hide
import apsw, apsw.fts5
from litesearch.core import _FTS_TOKENIZE
_c = apsw.Connection(':memory:')
apsw.fts5.register_tokenizers(_c, apsw.fts5.map_tokenizers); register_sanskrit(_c)
_c.execute(f"CREATE VIRTUAL TABLE _t USING fts5(c, tokenize='{_FTS_TOKENIZE}')")
for _r in ['श्रीमाता चिदग्नि-कुण्ड-सम्भूता दे॒वी', 'धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः',
           'The cats were running and fts_search stayed whole']:
    _c.execute('INSERT INTO _t(c) VALUES(?)', (_r,))
def _hits(q): return _c.execute('SELECT count(*) FROM _t WHERE _t MATCH ?', (q,)).fetchone()[0]
# cross-script and accent-blind
assert _hits('श्रीमाता') and _hits('srimata') and _hits('śrīmātā') and _hits('देवी')
# and English is untouched: porter still stems, identifiers still survive
assert _hits('running') and _hits('cat') and _hits('fts_search')

# Regression: the fold has to be emitted *inside* porter. With `sanskrit` wrapping the chain the
# fold was computed on porter's output. Devanagari passes porter untouched, so the index held the
# unstemmed `dharmaksetre` while the query for the same word stemmed to `dharmaksetr` and missed.
# `srimata` is unaffected (porter leaves it alone), which is exactly why the old test passed.
# These are locatives in `-e`, the ending that makes the mismatch routine in Sanskrit.
for _q in ['dharmaksetre', 'kuruksetre', 'dharmakṣetre']:
    assert _hits(_q) == 1, f'{_q!r} should reach the Devanagari row without a wildcard'
# and the exact, unwildcarded query is what has to work: a prefix search hid this bug
assert _hits('"dharmaksetre"') == 1

## Citations

`CITE_RE` matches a GRETIL citation, `// Mn_1.1 //` or `|| BrhUp_1,1.2 ||`. It lives here because `tree.py` builds the document tree out of it: the citation is the verse's address, so a Sanskrit source gets a real tree with no markup at all.

In [ ]:
#| export
CITE_RE = re.compile(r'(?:\|\||//)\s*([A-Za-zĀ-ſ]+)[_\s.]([\d,.\-]+[a-z]?(?:\[[^\]\n]{0,16}\])?)\s*(?:\|\||//)')
def cite_parts(c:str) -> tuple:
    "`'BrhUp_1,1.2'` -> `('BrhUp', ['1','1','2'])` — the siglum and its hierarchy."
    c = re.sub(r'\[[^\]]*\]', '', (c or '')).strip()   # drop a `[57M]` alternate-edition number
    m = re.match(r'([^\s_]+)[_\s]([\d,.\-]+[a-z]?)$', c) or re.match(r'([A-Za-zĀ-ſ]+)[.]([\d,.\-]+[a-z]?)$', c)
    if not m: return (c, [])
    return m.group(1), [p for p in re.split(r'[,.]', m.group(2)) if p]

In [ ]:
from litesearch.sanskrit import cite_parts

assert cite_parts('BrhUp_1,1.2') == ('BrhUp', ['1','1','2'])
assert cite_parts('Mn_1.1') == ('Mn', ['1','1'])
assert cite_parts('IsUp_4[57M]') == ('IsUp', ['4'])      # the alternate-edition number is dropped
assert cite_parts('not a citation') == ('not a citation', [])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()